# Lister les buckets et fichiers dans MinIO

Ce notebook vous montre comment **se connecter à un serveur MinIO** et **lister tous les buckets et fichiers** qui y sont stockés.

## Qu'est-ce que MinIO ?

MinIO est un serveur de stockage objet open-source compatible avec Amazon S3.
Imaginez un serveur de fichiers dans le cloud où les fichiers sont organisés dans des **buckets** (comme des dossiers).

## Prérequis

- Le package Python `minio` doit être installé (`pip install minio`)
- Les **variables d'environnement** suivantes doivent être définies sur votre système :
  - `MINIO_ENDPOINT` — l'adresse du serveur (ex. `minio.example.com:9000`)
  - `MINIO_ACCESS_KEY` — votre clé d'accès (comme un nom d'utilisateur)
  - `MINIO_SECRET_KEY` — votre clé secrète (comme un mot de passe)
  - `MINIO_SECURE` — optionnel, mettre à `true` pour utiliser HTTPS (par défaut `false`)

## Étape 1 — Installer la bibliothèque MinIO

Exécutez cette cellule pour vous assurer que le package `minio` est installé.

In [ ]:
# Installer la bibliothèque client Python pour MinIO
# Le préfixe '!' exécute une commande shell depuis un notebook
!pip install minio --quiet

## Étape 2 — Importer les bibliothèques

Nous avons besoin de deux bibliothèques :
- `os` — pour lire les variables d'environnement du système
- `minio` — pour communiquer avec le serveur MinIO

In [ ]:
import os            # Bibliothèque intégrée pour accéder aux variables d'environnement
from minio import Minio  # Bibliothèque client MinIO

## Étape 3 — Lire les paramètres de connexion depuis les variables d'environnement

Les variables d'environnement gardent les informations sensibles (mots de passe, clés) **en dehors de votre code**.
Elles sont définies sur le système et lues au moment de l'exécution, vous n'avez donc jamais à écrire de secrets dans un notebook.

| Variable           | Description                                        |
|--------------------|----------------------------------------------------|
| `MINIO_ENDPOINT`   | Adresse du serveur, ex. `minio.example.com:9000`   |
| `MINIO_ACCESS_KEY` | Votre clé d'accès (comme un nom d'utilisateur)     |
| `MINIO_SECRET_KEY` | Votre clé secrète (comme un mot de passe)          |
| `MINIO_SECURE`     | `true` pour HTTPS, `false` pour HTTP (par défaut)  |

In [ ]:
# Lire les paramètres de connexion depuis les variables d'environnement.
# os.environ["VAR"] lève une erreur si la variable est absente — c'est
# intentionnel pour que vous remarquiez tout de suite si quelque chose
# n'est pas configuré.

endpoint   = os.environ["MINIO_ENDPOINT"]    # ex. "minio.example.com:9000"
access_key = os.environ["MINIO_ACCESS_KEY"]   # votre clé d'accès
secret_key = os.environ["MINIO_SECRET_KEY"]   # votre clé secrète

# Pour le paramètre secure, on utilise os.environ.get() qui retourne une
# valeur par défaut au lieu de lever une erreur quand la variable n'existe pas.
secure     = os.environ.get("MINIO_SECURE", "false").lower() == "true"

print(f"Point d'accès MinIO : {endpoint}")
print(f"Sécurisé (HTTPS)    : {secure}")

## Étape 4 — Se connecter à MinIO

Créer un objet **client**. Cela n'envoie pas encore de requête — cela prépare simplement la connexion.

In [ ]:
# Créer un client MinIO avec les paramètres lus ci-dessus.
# Le client gère toute la communication avec le serveur MinIO.

client = Minio(
    endpoint,               # adresse du serveur
    access_key=access_key,   # authentification : qui vous êtes
    secret_key=secret_key,   # authentification : preuve d'identité
    secure=secure,           # True = HTTPS, False = HTTP
)

print("Client créé avec succès — prêt à se connecter !")

## Étape 5 — Lister tous les buckets

Un **bucket** est un conteneur de premier niveau dans le stockage objet (similaire à un dossier).
Récupérons et affichons tous les buckets disponibles sur le serveur.

In [ ]:
# list_buckets() contacte le serveur et retourne la liste de tous les buckets
# que votre clé d'accès est autorisée à voir.

buckets = client.list_buckets()

print(f"{len(buckets)} bucket(s) trouvé(s) :\n")

for bucket in buckets:
    # Chaque bucket a un nom (name) et une date de création (creation_date)
    print(f"  - {bucket.name}  (créé le : {bucket.creation_date})")

## Étape 6 — Lister les fichiers dans chaque bucket

Maintenant, nous parcourons chaque bucket et listons les **objets** (fichiers) qu'il contient.

`list_objects()` retourne un itérateur — nous parcourons chaque élément et affichons son nom et sa taille.

In [ ]:
for bucket in buckets:
    print(f"\n{'='*60}")
    print(f"Bucket : {bucket.name}")
    print(f"{'='*60}")

    # list_objects() liste tous les objets dans le bucket.
    # recursive=True signifie qu'on liste aussi les objets dans les « sous-dossiers ».
    objects = client.list_objects(bucket.name, recursive=True)

    count = 0
    for obj in objects:
        # obj.object_name  = chemin complet du fichier
        # obj.size         = taille du fichier en octets
        # obj.last_modified = date de dernière modification
        size_kb = (obj.size or 0) / 1024  # convertir les octets en kilo-octets
        print(f"  {obj.object_name:<50} {size_kb:>10.1f} Ko   {obj.last_modified}")
        count += 1

    if count == 0:
        print("  (bucket vide — aucun fichier trouvé)")
    else:
        print(f"  --- {count} fichier(s) au total ---")

## Résumé

Dans ce notebook, vous avez appris à :

1. **Lire les paramètres de connexion** depuis les variables d'environnement (pour garder les secrets hors du code)
2. **Créer un client MinIO** pour communiquer avec le serveur
3. **Lister tous les buckets** sur le serveur
4. **Lister tous les fichiers** dans chaque bucket

### Prochaines étapes

- Envoyer un fichier : `client.fput_object("mon-bucket", "nom-distant.csv", "/chemin/local.csv")`
- Télécharger un fichier : `client.fget_object("mon-bucket", "nom-distant.csv", "/chemin/local.csv")`
- Créer un bucket : `client.make_bucket("nouveau-bucket")`